In [ ]:
import os
import requests
import json
import urllib3

urllib3.disable_warnings()

os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""
os.environ["all_proxy"] = ""

project_dir = "AI_Quote_Generator"
if os.path.basename(os.getcwd()) == project_dir:
    project_dir = "."

templates_dir = os.path.join(project_dir, "templates")
static_dir = os.path.join(project_dir, "static")

os.makedirs(templates_dir, exist_ok=True)
os.makedirs(static_dir, exist_ok=True)

print("Folder structure initialized successfully")


In [ ]:
API_KEY = "YOUR_API_KEY_HERE"
API_URL = "https://api.apifree.ai/v1/chat/completions"
MODEL_NAME = "deepseek-ai/deepseek-v3.2"

def ask_ai(prompt):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "User-Agent": "Mozilla/5.0"
    }

    data = {
        "max_tokens": 8192,
        "messages": [
            {"content": prompt, "role": "user"}
        ],
        "model": MODEL_NAME,
        "stream": False,
        "temperature": 1,
        "top_p": 1
    }

    try:
        response = requests.post(
            API_URL,
            headers=headers,
            json=data,
            verify=False,
            proxies={"http": "", "https": ""},
            timeout=60
        )

        response.raise_for_status()
        result = response.json()
        text = result["choices"][0]["message"]["content"]

        text = text.replace("```python", "")
        text = text.replace("```html", "")
        text = text.replace("```css", "")
        text = text.replace("```javascript", "")
        text = text.replace("```js", "")
        text = text.replace("```plantuml", "")
        text = text.replace("```", "")

        return text.strip()

    except Exception as e:
        print("Request failed:", e)
        return None


In [ ]:
print("Generating app.py ...")
app_code = ask_ai("You are a senior Python engineer. Write a Flask backend app.py. It must include a root route '/' that renders index.html, a '/api/quote' route that returns a random inspirational quote as JSON, and a '/generated-image' route that retrieves an automatically generated image from Pollinations API. Return only pure Python code without explanation.")

if app_code is None:
    print("app.py generation failed. Please check API_KEY, network connection, or API_URL.")
else:
    with open(os.path.join(project_dir, "app.py"), "w", encoding="utf-8") as file:
        file.write(app_code)

    print("Generating index.html ...")
    html_code = ask_ai("Write a frontend index.html. It should display an automatically generated image from Pollinations API or from the Flask '/generated-image' route, show a quote text below it, and include a button. When the button is clicked, JavaScript should fetch '/api/quote' and update the quote text. Return only pure HTML code without explanation.")

    if html_code is None:
        print("index.html generation failed. Please check API_KEY, network connection, or API_URL.")
    else:
        with open(os.path.join(project_dir, "templates", "index.html"), "w", encoding="utf-8") as file:
            file.write(html_code)

        print("Generating UML diagram ...")
        uml_code = ask_ai("Write a PlantUML sequence diagram in English. Describe this process: user opens the web page, frontend requests the Flask backend, backend returns the page, frontend displays an automatically generated image, user clicks the quote button, frontend requests /api/quote, Flask backend returns JSON data, and frontend updates the quote text. Return only the content from @startuml to @enduml.")

        if uml_code is None:
            print("UML diagram generation failed. Please check API_KEY, network connection, or API_URL.")
        else:
            with open(os.path.join(project_dir, "diagram.puml"), "w", encoding="utf-8") as file:
                file.write(uml_code)

            print("All files generated successfully")
